# Bonus — Generative Models — Self-Study Exploration

This notebook is optional. Work through it at your own pace after the school, in any order.
There are no exercises and no `raise NotImplementedError` stubs — this is pure exploration.

**What is covered:**

1. **Variational Autoencoders (VAEs)** — latent spaces, the reparameterization trick, and how a VAE differs from a plain autoencoder
2. **Generative Adversarial Networks (GANs)** — the minimax game, training instability, and watching a generator distribution evolve
3. **Diffusion models (conceptual)** — the forward noising process visualised step by step; why the reverse process is remarkable
4. **Self-supervised / contrastive learning** — SimCLR intuition, augmented views, and how embeddings cluster after contrastive training

---

## Where this fits

| # | Topic | Slide deck | Notebook |
|---|---|---|---|
| 1 | Single neuron | `01_single_neuron.pdf` | `01_single_neuron.ipynb` |
| 2 | Multilayer networks | `02_multilayer_networks.pdf` | `02_training_loop.ipynb` |
| 3 | Backpropagation | `03_backprop_training.pdf` | `03_backpropagation.ipynb` |
| 4 | Optimizers | `07_optimizers.pdf` *(new)* | `04_optimizers.ipynb` |
| 5 | CNNs | `04_cnns.pdf` | `05_cnns.ipynb` |
| **→ 6** | **Modern architectures** | **`05a_attention.pdf` + `05b_practical.pdf`** | ***(bonus: `bonus_generative_models.ipynb`)*** |
| 7 | Bayesian inference | `06_bayesian_inference.pdf` | `06_bayesian_inference.ipynb` |

**Coming from:** Decks 04 and 05 covered discriminative networks (classification, regression); this notebook flips to models that *generate* data.

**Leading to:** Nothing — open this any time after the school as pure exploration, in any order.

**If you skipped ahead:** Read decks 04 and 05 first; this notebook assumes you already know what a CNN and attention are.


## Setup

Run this cell first. On Colab it installs the required packages; on JupyterHub they are already available.

In [ ]:
# ── Environment setup ──────────────────────────────────────────────────────
# Run this cell only on Colab. On JupyterHub, packages are pre-installed.
import sys
if 'google.colab' in sys.modules:
    %pip install -q ipywidgets torch

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from ipywidgets import interact, IntSlider, FloatSlider
%matplotlib inline

---

## Part 1 — Variational Autoencoders

A plain autoencoder compresses input data to a latent code `z` and reconstructs it.
The problem: the latent space has no structure — codes cluster arbitrarily, and sampling a random point in between clusters produces garbage.
A **Variational Autoencoder** (Kingma & Welling, 2013) fixes this by training the encoder to produce a *distribution* over `z` rather than a single point.
The encoder outputs a mean `\mu` and log-variance `\log\sigma^2`; a sample is drawn as `z = \mu + \sigma \cdot \epsilon` where `\epsilon \sim \mathcal{N}(0, I)` — this is the **reparameterization trick** that makes the sampling step differentiable.

The training objective balances two terms:
$$\mathcal{L} = \underbrace{\|x - \hat{x}\|^2}_{\text{reconstruction}} + \underbrace{D_{KL}(q(z|x) \| \mathcal{N}(0,I))}_{\text{regularisation}}$$

The KL term pushes the encoder distribution toward a standard normal, which makes the latent space smooth and interpolatable.

### Toy dataset: three galaxy types

We generate a 2D toy dataset representing three galaxy populations — each a Gaussian cluster in a higher-dimensional observation space (here 8D, so each 'galaxy' has 8 photometric features).
We will train a VAE with a **2D latent space** so we can visualise the entire latent space in a scatter plot and slide through it interactively.

In [ ]:
rng = np.random.RandomState(42)

# Three galaxy 'types' in 8D observation space
n_per_class = 200
centers = np.array([
    [2.0,  1.0,  0.5,  0.2, -0.5, -1.0,  0.3,  0.1],   # elliptical
    [-1.5, 0.5, -0.5,  1.5,  0.5,  0.2, -0.3,  0.8],   # spiral
    [0.0, -2.0,  1.0, -1.0,  1.5, -0.5,  0.9, -0.6],   # irregular
])
labels_list = []
data_list = []
for i, c in enumerate(centers):
    x = rng.randn(n_per_class, 8) * 0.6 + c
    data_list.append(x)
    labels_list.extend([i] * n_per_class)

X_np = np.vstack(data_list).astype(np.float32)
y_np = np.array(labels_list)

X_tensor = torch.from_numpy(X_np)
print(f'Dataset shape: {X_np.shape}   (600 galaxies, 8 features)')
print(f'Classes: 0=elliptical, 1=spiral, 2=irregular')

### VAE architecture

The encoder maps 8D inputs to a 2D latent distribution `(mu, logvar)`.
The decoder maps a 2D latent sample back to 8D.
This is adapted directly from `lecture4_demo.py` and expanded to a 2D latent space for visualisation.

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim=8, hidden_dim=32, latent_dim=2):
        super().__init__()
        # Encoder
        self.enc_fc1  = nn.Linear(input_dim, hidden_dim)
        self.enc_mu   = nn.Linear(hidden_dim, latent_dim)
        self.enc_logv = nn.Linear(hidden_dim, latent_dim)
        # Decoder
        self.dec_fc1  = nn.Linear(latent_dim, hidden_dim)
        self.dec_out  = nn.Linear(hidden_dim, input_dim)

    def encode(self, x):
        h = torch.relu(self.enc_fc1(x))
        return self.enc_mu(h), self.enc_logv(h)

    def reparameterize(self, mu, logvar):
        # Reparameterization trick: z = mu + sigma * eps
        # This keeps the gradient flowing through mu and logvar
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = torch.relu(self.dec_fc1(z))
        return self.dec_out(h)  # no sigmoid: MSE loss on raw values

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar


def vae_loss(recon, x, mu, logvar):
    recon_loss = nn.functional.mse_loss(recon, x, reduction='sum')
    # KL divergence: -0.5 * sum(1 + logvar - mu^2 - exp(logvar))
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl


torch.manual_seed(0)
vae = VAE()
print(vae)

### Training the VAE

Training takes a few seconds on CPU. We store the reconstruction loss and KL term separately at each epoch so we can inspect the balance between the two objectives.

In [ ]:
torch.manual_seed(0)
vae = VAE()
optimizer = optim.Adam(vae.parameters(), lr=1e-3)

n_epochs = 300
losses = []

for epoch in range(n_epochs):
    optimizer.zero_grad()
    recon, mu, logvar = vae(X_tensor)
    loss = vae_loss(recon, X_tensor, mu, logvar)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

plt.figure(figsize=(7, 3))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Total VAE loss')
plt.title('VAE training')
plt.grid(True)
plt.tight_layout()
plt.show()

### Latent space scatter plot

After training, we pass all 600 galaxies through the encoder and plot where they land in the 2D latent space, coloured by type.
Notice that the three clusters are separated but the boundaries are smooth — unlike a plain autoencoder, the VAE has been regularised so that the space between clusters is also meaningful.

In [ ]:
vae.eval()
with torch.no_grad():
    mu_all, _ = vae.encode(X_tensor)
mu_all = mu_all.numpy()

colours = ['#e76f51', '#2a9d8f', '#457b9d']
labels_str = ['elliptical', 'spiral', 'irregular']

fig, ax = plt.subplots(figsize=(6, 5))
for i in range(3):
    mask = y_np == i
    ax.scatter(mu_all[mask, 0], mu_all[mask, 1],
               c=colours[i], label=labels_str[i], alpha=0.6, s=18)
ax.set_xlabel('Latent dim 1')
ax.set_ylabel('Latent dim 2')
ax.set_title('VAE latent space (encoder means)')
ax.legend()
ax.grid(True, linewidth=0.4)
plt.tight_layout()
plt.show()

### Widget: slide through the latent space

Use the sliders to set a point `(z1, z2)` anywhere in the latent space and watch the decoder reconstruct the corresponding 8D feature vector.
Try dragging from the centre of one cluster toward another — you are interpolating smoothly between galaxy types, which would not be possible with a plain autoencoder.

In [ ]:
def explore_latent(z1, z2):
    z = torch.tensor([[z1, z2]], dtype=torch.float32)
    with torch.no_grad():
        decoded = vae.decode(z).numpy().flatten()

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    # Left: latent space with current point marked
    ax = axes[0]
    for i in range(3):
        mask = y_np == i
        ax.scatter(mu_all[mask, 0], mu_all[mask, 1],
                   c=colours[i], label=labels_str[i], alpha=0.3, s=10)
    ax.scatter([z1], [z2], c='black', s=120, zorder=5, marker='*', label='current z')
    ax.set_xlabel('Latent dim 1')
    ax.set_ylabel('Latent dim 2')
    ax.set_title('Latent space')
    ax.legend(fontsize=8)
    ax.grid(True, linewidth=0.4)

    # Right: decoded feature vector as a bar chart
    ax = axes[1]
    ax.bar(range(8), decoded, color='#457b9d', alpha=0.8)
    ax.set_xticks(range(8))
    ax.set_xticklabels([f'f{i}' for i in range(8)])
    ax.set_ylabel('Feature value')
    ax.set_title(f'Decoded output at z = ({z1:.2f}, {z2:.2f})')
    ax.grid(True, axis='y', linewidth=0.4)

    plt.tight_layout()
    plt.show()

interact(
    explore_latent,
    z1=FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.1, description='z1'),
    z2=FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.1, description='z2'),
);

### Think about it

- Move `z1` and `z2` to a position well outside the three clusters (e.g. `z1=2.5, z2=2.5`). What does the decoder output look like? Is it meaningful?
- The KL regularisation term pushes all latent distributions toward `N(0, I)`. What would happen to the latent space if you removed that term entirely and trained a plain autoencoder?
- In the scatter plot, are the three galaxy classes cleanly separated or do they overlap? What would tighter separation cost in terms of the reconstruction loss?
- Astronomers use VAEs to find anomalies: sources that encode to low-probability regions of the latent space may be rare or unusual. Where in the latent scatter plot would you start looking for outliers?

---

## Part 2 — Generative Adversarial Networks

A GAN (Goodfellow et al., 2014) trains two networks simultaneously: a **generator** G that maps random noise to synthetic data, and a **discriminator** D that tries to tell real from fake.
The training objective is a minimax game: G minimises `log(1 - D(G(z)))` while D maximises `log D(x) + log(1 - D(G(z)))`.
At equilibrium (rarely reached in practice), G produces data indistinguishable from real, and D outputs 0.5 everywhere.
In practice, GAN training is notoriously unstable — the losses oscillate, and the balance between G and D is fragile.

> **Note:** The implementation below is a minimal sketch intended to illustrate the dynamics. Real-world GAN training requires careful hyperparameter tuning, gradient penalties, and architectural choices (e.g. DCGAN, Wasserstein GAN) that are beyond scope here.

### Toy dataset: 1D mixture of Gaussians

We use a 1D target distribution made of two Gaussians, representing a bimodal spectral line distribution (e.g. two populations at different redshifts).
The generator takes 1D noise and should learn to produce samples from this bimodal distribution after training.

In [ ]:
rng_gan = np.random.RandomState(7)

def sample_real(n, rng=rng_gan):
    '''Sample from a bimodal Gaussian: two spectral populations.'''
    half = n // 2
    a = rng.randn(half) * 0.3 - 1.5
    b = rng.randn(n - half) * 0.3 + 1.5
    return np.concatenate([a, b]).astype(np.float32)

# Preview the target distribution
real_preview = sample_real(2000)
plt.figure(figsize=(7, 3))
plt.hist(real_preview, bins=60, density=True, alpha=0.7, color='#2a9d8f', label='real')
plt.xlabel('Spectral feature value')
plt.ylabel('Density')
plt.title('Target distribution (mixture of two Gaussians)')
plt.legend()
plt.grid(True, linewidth=0.4)
plt.tight_layout()
plt.show()

### GAN architecture and training

We train for 2000 steps and record the generator and discriminator losses at each step.
We also store snapshots of the generator's output distribution every 200 steps so we can watch it evolve.

In [ ]:
class Generator1D(nn.Module):
    def __init__(self, noise_dim=1, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1),
        )
    def forward(self, z):
        return self.net(z)


class Discriminator1D(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden, hidden),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden, 1),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return self.net(x)


torch.manual_seed(1)
G = Generator1D()
D = Discriminator1D()
opt_G = optim.Adam(G.parameters(), lr=1e-3)
opt_D = optim.Adam(D.parameters(), lr=1e-3)
bce = nn.BCELoss()

n_steps = 2000
batch_size = 128
snapshot_every = 200

g_losses, d_losses = [], []
snapshots = {}  # step -> generated samples

rng_train = np.random.RandomState(42)

for step in range(n_steps):
    # ── Train Discriminator ──
    real = torch.from_numpy(sample_real(batch_size, rng_train)).unsqueeze(1)
    z = torch.randn(batch_size, 1)
    fake = G(z).detach()

    d_real = D(real)
    d_fake = D(fake)
    loss_D = bce(d_real, torch.ones_like(d_real)) + bce(d_fake, torch.zeros_like(d_fake))

    opt_D.zero_grad()
    loss_D.backward()
    opt_D.step()

    # ── Train Generator ──
    z = torch.randn(batch_size, 1)
    fake = G(z)
    d_fake = D(fake)
    loss_G = bce(d_fake, torch.ones_like(d_fake))  # G wants D to predict 'real'

    opt_G.zero_grad()
    loss_G.backward()
    opt_G.step()

    g_losses.append(loss_G.item())
    d_losses.append(loss_D.item())

    if step % snapshot_every == 0 or step == n_steps - 1:
        with torch.no_grad():
            samples = G(torch.randn(2000, 1)).numpy().flatten()
        snapshots[step] = samples

print(f'Training complete. Steps: {n_steps}')

### Loss curves

The characteristic signature of GAN training: G loss and D loss oscillate throughout, never converging to a stable minimum in the way a supervised loss curve does.
When D loss drops very low, D has learned to easily distinguish real from fake — the generator has fallen behind.
When G loss drops, the generator is temporarily fooling D.

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(g_losses, alpha=0.7, label='G loss', color='#e76f51')
plt.plot(d_losses, alpha=0.7, label='D loss', color='#457b9d')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('GAN training losses — note the oscillation')
plt.legend()
plt.grid(True, linewidth=0.4)
plt.tight_layout()
plt.show()

### Widget: epoch slider — watch the generator distribution evolve

Slide through the training snapshots to see how the generator's output distribution changes.
Early on, the generator produces a narrow Gaussian near zero — it has not yet learned the bimodal structure.
Later, the distribution spreads and, hopefully, develops two modes.

In [ ]:
snapshot_steps = sorted(snapshots.keys())

def show_gan_snapshot(step_idx):
    step = snapshot_steps[step_idx]
    gen_samples = snapshots[step]

    plt.figure(figsize=(7, 3))
    plt.hist(real_preview, bins=60, density=True, alpha=0.5,
             color='#2a9d8f', label='real')
    plt.hist(gen_samples, bins=60, density=True, alpha=0.5,
             color='#e76f51', label=f'generated (step {step})')
    plt.xlabel('Value')
    plt.ylabel('Density')
    plt.title(f'Generator distribution at training step {step}')
    plt.legend()
    plt.xlim(-4, 4)
    plt.grid(True, linewidth=0.4)
    plt.tight_layout()
    plt.show()

interact(
    show_gan_snapshot,
    step_idx=IntSlider(value=0, min=0, max=len(snapshot_steps)-1, step=1,
                       description='snapshot'),
);

### Think about it

- At which snapshot does the generator first start developing two modes? Does it capture both modes simultaneously or one before the other?
- Look at the loss curves around that transition. Does the D loss increase or decrease when G first discovers the second mode?
- A known failure mode of GANs is **mode collapse**: the generator finds a single high-density region it can reliably fool D with and stops exploring. Does this happen here? What would the histogram look like if it did?
- The GAN here is trained on a 1D distribution with 128 samples per step for 2000 steps. In astronomy, a GAN might be trained on 50,000 galaxy images with millions of parameters. What does that imply about the difficulty of diagnosing and fixing mode collapse in a real application?

---

## Part 3 — Diffusion Models (Conceptual)

Diffusion models (Ho et al., 2020) define a **forward process** that gradually adds Gaussian noise to data over `T` steps until only noise remains.
A neural network is then trained to reverse this process — to predict the noise added at each step, so it can iteratively denoise a sample back to a clean image.
The forward process is fixed and has no learnable parameters; all the learning is in the reverse model.

The forward process at step `t` is:
$$x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$
where `\bar{\alpha}_t = \prod_{s=1}^{t}(1 - \beta_s)` is the cumulative noise factor and `\beta_s` follows a linear schedule from a small value near 0 to a value near 1.

> **What to appreciate here:** training a neural network to precisely reverse this noising process, across all `T` steps, well enough to generate photorealistic images from pure noise — is remarkable. The widget below shows only the forward direction.

### Synthetic galaxy image

We generate a simple synthetic galaxy: a Gaussian bulge plus an exponential disc, evaluated on a 64x64 pixel grid.
This is purely for demonstration — the same principle applies to real FITS images.

In [ ]:
rng_diff = np.random.RandomState(3)

def make_galaxy(size=64):
    '''Synthetic galaxy: Gaussian bulge + exponential disc on a size x size grid.'''
    cy, cx = size / 2, size / 2
    y, x = np.mgrid[0:size, 0:size].astype(np.float32)
    r = np.sqrt((x - cx)**2 + (y - cy)**2)
    bulge = np.exp(-r**2 / 0.4 / (size / 8)**2)
    disc  = np.exp(-r / (size / 6))
    img = 0.6 * bulge + 0.4 * disc
    img = img / img.max()
    return img.astype(np.float32)

galaxy = make_galaxy(64)

plt.figure(figsize=(3.5, 3.5))
plt.imshow(galaxy, cmap='inferno', origin='lower')
plt.colorbar(label='Normalised flux')
plt.title('Synthetic galaxy (x_0)')
plt.axis('off')
plt.tight_layout()
plt.show()

### Noise schedule

We use a **linear beta schedule**: `beta_t` increases linearly from `beta_start` to `beta_end` over `T` steps.
From this we compute `alpha_bar_t = prod(1 - beta_s for s <= t)`, which is the mixing coefficient in the forward process formula.
As `t` increases, `alpha_bar_t` falls toward 0 and the image becomes dominated by noise.

In [ ]:
T = 200
beta_start = 1e-4
beta_end   = 0.02

betas      = np.linspace(beta_start, beta_end, T, dtype=np.float32)
alphas     = 1.0 - betas
alpha_bar  = np.cumprod(alphas)  # alpha_bar_t = prod_{s=1}^{t} alpha_s

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(betas, color='#457b9d')
axes[0].set_xlabel('Step t')
axes[0].set_ylabel('beta_t')
axes[0].set_title('Linear noise schedule (beta)')
axes[0].grid(True, linewidth=0.4)

axes[1].plot(alpha_bar, color='#e76f51')
axes[1].set_xlabel('Step t')
axes[1].set_ylabel('alpha_bar_t')
axes[1].set_title('Cumulative signal fraction')
axes[1].axhline(0.5, color='grey', linestyle='--', linewidth=0.8, label='0.5')
axes[1].legend()
axes[1].grid(True, linewidth=0.4)

plt.tight_layout()
plt.show()

half_signal_step = np.argmin(np.abs(alpha_bar - 0.5))
print(f'Signal fraction falls below 0.5 at step t = {half_signal_step}')

### Widget: forward noising process

Slide through steps 0 to T to watch the galaxy progressively disappear into noise.
The left panel shows the noisy image `x_t`; the right panel shows the pixel-value histogram so you can see the distribution shifting from a peaked galaxy profile toward a standard Gaussian.

In [ ]:
rng_noise = np.random.RandomState(99)
# Pre-draw one noise realisation so the widget is deterministic across slider moves
fixed_eps = rng_noise.randn(*galaxy.shape).astype(np.float32)

def show_diffusion_step(t):
    if t == 0:
        noisy = galaxy.copy()
    else:
        ab = alpha_bar[t - 1]  # alpha_bar at step t (0-indexed)
        noisy = np.sqrt(ab) * galaxy + np.sqrt(1.0 - ab) * fixed_eps

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))

    im = axes[0].imshow(noisy, cmap='inferno', origin='lower',
                        vmin=-2.5, vmax=2.5)
    axes[0].set_title(f'x_t  (t = {t} / {T})')
    axes[0].axis('off')
    plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

    axes[1].hist(noisy.flatten(), bins=60, density=True,
                 color='#457b9d', alpha=0.8)
    xs = np.linspace(-3.5, 3.5, 200)
    axes[1].plot(xs, np.exp(-0.5 * xs**2) / np.sqrt(2 * np.pi),
                 'r--', linewidth=1.5, label='N(0,1)')
    axes[1].set_xlabel('Pixel value')
    axes[1].set_ylabel('Density')
    axes[1].set_title('Pixel distribution')
    axes[1].legend()
    axes[1].grid(True, linewidth=0.4)

    ab_val = alpha_bar[t - 1] if t > 0 else 1.0
    fig.suptitle(
        f'alpha_bar = {ab_val:.4f}   '
        f'({100*ab_val:.1f}% signal, {100*(1-ab_val):.1f}% noise)',
        y=1.01
    )
    plt.tight_layout()
    plt.show()

interact(
    show_diffusion_step,
    t=IntSlider(value=0, min=0, max=T, step=1, description='step t'),
);

### Think about it

- At what step `t` does the galaxy become visually unrecognisable? Does that match the step where `alpha_bar` crosses 0.5?
- The pixel histogram approaches `N(0,1)` (the red dashed line) as `t` increases. By step `T`, do the two curves match? What does that tell you about what a diffusion model samples from at inference time?
- A cosine noise schedule (used in improved DDPM) decays `alpha_bar` more slowly at the beginning and end. Looking at the linear schedule plot, why might this be beneficial for learning the reverse process?
- In astronomy, diffusion models have been proposed for denoising radio telescope images (thermal noise, RFI). The forward process is defined mathematically — but the actual noise in a telescope image is different from Gaussian. What would you need to change?

---

## Part 4 — Self-supervised / Contrastive Learning

Supervised learning requires labelled data, which in astronomy is expensive: galaxy morphology labels, spectroscopic redshifts, and transient classifications all require expert time.
**Self-supervised learning** creates a supervision signal from the data itself, without any labels.
In SimCLR (Chen et al., 2020), two randomly augmented views of the same image are treated as a **positive pair**; views of different images are **negative pairs**.
The model is trained to pull positive pairs together and push negative pairs apart in an embedding space.

The loss used is **InfoNCE** (also called NT-Xent in SimCLR):
$$\mathcal{L}_i = -\log \frac{\exp(\text{sim}(z_i, z_j) / \tau)}{\sum_{k \neq i} \exp(\text{sim}(z_i, z_k) / \tau)}$$
where `\tau` is a temperature hyperparameter and `sim` is cosine similarity.

> **Note on the source code:** `ContrastiveLoss` in `lecture4_demo.py` is a non-standard approximation that does not correctly implement NT-Xent — in particular it does not properly form the full 2N×2N similarity matrix. We implement a clean InfoNCE loss below instead.

### Toy dataset: three galaxy morphology clusters

We use 2D points from three clusters, representing three galaxy morphologies.
Augmentation here means adding small Gaussian perturbations to each point — a minimal stand-in for the cropping, colour jitter, and blur used in image-based SimCLR.

In [ ]:
rng_cl = np.random.RandomState(17)

# Three clusters in 2D input space
n_cl = 150
cl_centers = np.array([[-3.0, 0.0], [3.0, 0.0], [0.0, 3.5]], dtype=np.float32)
cl_labels = []
cl_data = []
for i, c in enumerate(cl_centers):
    pts = rng_cl.randn(n_cl, 2).astype(np.float32) * 0.7 + c
    cl_data.append(pts)
    cl_labels.extend([i] * n_cl)

X_cl = np.vstack(cl_data)
y_cl = np.array(cl_labels)
X_cl_t = torch.from_numpy(X_cl)

def augment(x_np, rng, noise_std=0.3):
    '''Simple augmentation: add Gaussian noise (stand-in for image augmentation).'''
    return x_np + rng.randn(*x_np.shape).astype(np.float32) * noise_std

plt.figure(figsize=(4.5, 4))
cl_colours = ['#e76f51', '#2a9d8f', '#457b9d']
cl_labels_str = ['elliptical', 'spiral', 'irregular']
for i in range(3):
    m = y_cl == i
    plt.scatter(X_cl[m, 0], X_cl[m, 1],
                c=cl_colours[i], label=cl_labels_str[i], alpha=0.5, s=14)
plt.title('Raw 2D input data')
plt.legend(fontsize=8)
plt.grid(True, linewidth=0.4)
plt.tight_layout()
plt.show()

### InfoNCE loss and contrastive training

We train a small MLP encoder (2D → 16D) with an InfoNCE objective.
At each step we produce two augmented views of the full dataset; the loss treats each sample's pair of views as positive and all other samples as negatives.
After training, we compare the embeddings before and after to see how well the three morphology clusters are separated.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=32, embed_dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim),
        )
    def forward(self, x):
        return nn.functional.normalize(self.net(x), dim=1)  # L2-normalise


def infonce_loss(z_i, z_j, temperature=0.1):
    '''
    InfoNCE / NT-Xent loss for a batch of N positive pairs (z_i[k], z_j[k]).
    z_i and z_j are L2-normalised embeddings of shape (N, D).
    All 2N samples form the denominator; each sample treats its paired view
    as the only positive.
    '''
    N = z_i.shape[0]
    z = torch.cat([z_i, z_j], dim=0)          # (2N, D)
    sim = torch.matmul(z, z.T) / temperature  # (2N, 2N)

    # Mask out self-similarities on the diagonal
    mask = torch.eye(2 * N, dtype=torch.bool)
    sim.masked_fill_(mask, float('-inf'))

    # Positive pair indices: (k, k+N) and (k+N, k)
    pos_idx = torch.cat([
        torch.arange(N, 2 * N),   # for the first N rows, positive is at row+N
        torch.arange(0, N),        # for the second N rows, positive is at row-N
    ])

    log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True)
    loss = -log_prob[torch.arange(2 * N), pos_idx].mean()
    return loss


torch.manual_seed(5)
encoder = Encoder()

# Record embeddings before training
encoder.eval()
with torch.no_grad():
    emb_before = encoder(X_cl_t).numpy()

opt_enc = optim.Adam(encoder.parameters(), lr=1e-3)
rng_aug = np.random.RandomState(21)
n_steps_cl = 500
cl_losses = []

encoder.train()
for step in range(n_steps_cl):
    view_i = torch.from_numpy(augment(X_cl, rng_aug, noise_std=0.3))
    view_j = torch.from_numpy(augment(X_cl, rng_aug, noise_std=0.3))

    z_i = encoder(view_i)
    z_j = encoder(view_j)
    loss = infonce_loss(z_i, z_j, temperature=0.1)

    opt_enc.zero_grad()
    loss.backward()
    opt_enc.step()
    cl_losses.append(loss.item())

# Record embeddings after training
encoder.eval()
with torch.no_grad():
    emb_after = encoder(X_cl_t).numpy()

print(f'Final InfoNCE loss: {cl_losses[-1]:.4f}')

### Widget: embeddings before vs after contrastive training

We project the 16D embeddings to 2D using PCA for visualisation.
Drag the slider to compare the random initialisation (step 0) with the trained embeddings (step 1).
Notice how the three morphology classes, which were never given any labels during training, emerge as distinct clusters in the embedding space.

In [ ]:
def pca_2d(X):
    '''Manual PCA to 2D: zero-mean, then top-2 eigenvectors of covariance.'''
    X_c = X - X.mean(axis=0)
    cov = X_c.T @ X_c / (len(X_c) - 1)
    eigvals, eigvecs = np.linalg.eigh(cov)
    # eigh returns in ascending order; take last two
    vecs = eigvecs[:, -2:][:, ::-1]
    return X_c @ vecs

proj_before = pca_2d(emb_before)
proj_after  = pca_2d(emb_after)

def show_embeddings(state):
    proj = proj_before if state == 0 else proj_after
    title = 'Embeddings BEFORE contrastive training' if state == 0 \
            else 'Embeddings AFTER contrastive training'

    fig, ax = plt.subplots(figsize=(5.5, 5))
    for i in range(3):
        m = y_cl == i
        ax.scatter(proj[m, 0], proj[m, 1],
                   c=cl_colours[i], label=cl_labels_str[i], alpha=0.6, s=20)
    ax.set_xlabel('PCA dim 1')
    ax.set_ylabel('PCA dim 2')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, linewidth=0.4)
    plt.tight_layout()
    plt.show()

interact(
    show_embeddings,
    state=IntSlider(value=0, min=0, max=1, step=1,
                    description='0=before 1=after'),
);

### Think about it

- The model was never told which cluster each point belonged to. Where did the cluster structure come from?
- Try increasing the augmentation noise (`noise_std`) from 0.3 to 1.0 and re-running. What happens to the embedding quality? Is there a level of augmentation that is too strong?
- The temperature `tau=0.1` controls how sharply the loss penalises negative pairs close in embedding space. Try `tau=1.0` — does the clustering become tighter or looser?
- In a real survey like DESI or Rubin, you might have millions of galaxy spectra and only a few thousand expert morphology labels. How would you use contrastive pre-training followed by fine-tuning to make the most of both?

---

## Where next?

These four models represent the current frontier of generative and self-supervised learning. The canonical papers are short and readable:

### References

| | |
|---|---|
| Primary | Kingma & Welling (2013) — *Auto-Encoding Variational Bayes*. arXiv:1312.6114. [https://arxiv.org/abs/1312.6114](https://arxiv.org/abs/1312.6114) |
| Primary | Goodfellow et al. (2014) — *Generative Adversarial Nets*. NeurIPS. [https://arxiv.org/abs/1406.2661](https://arxiv.org/abs/1406.2661) |
| Primary | Ho, Jain & Abbeel (2020) — *Denoising Diffusion Probabilistic Models*. NeurIPS. [https://arxiv.org/abs/2006.11239](https://arxiv.org/abs/2006.11239) |
| Primary | Chen et al. (2020) — *A Simple Framework for Contrastive Learning of Visual Representations (SimCLR)*. ICML. [https://arxiv.org/abs/2002.05709](https://arxiv.org/abs/2002.05709) |
| Video | [Denoising Diffusion Probabilistic Models — Yannic Kilcher](https://www.youtube.com/watch?v=W-O7AZNzbzQ) |
| Blog | Lilian Weng — *What are Diffusion Models?* [https://lilianweng.github.io/posts/2021-07-11-diffusion-models/](https://lilianweng.github.io/posts/2021-07-11-diffusion-models/) |